# Comparing and Tuning Local LLMs for Health and Social Science Research

## Lab Four (Estimated Duration: ~2 hours)

This lab is **model-only**. It requires Ollama and locally downloaded model weights. The setup cells below can attempt an in-notebook Ollama install on Linux, pull the required model weights by default, verify that the models are visible to `ollama list`, and then use real local model calls for the comparison and tuning exercises.

There is no model-free path in this notebook. If Ollama cannot be installed or found, the models cannot be pulled, or the local server cannot answer requests, the relevant cell will raise an error that needs to be fixed before continuing.


## How to Use This Notebook

- Run cells from top to bottom.
- Expect the setup cells to take time: on Linux they may install Ollama, and they may download several GB of model weights.
- Use only the synthetic health and social examples in this notebook.
- Do not paste confidential, patient, client, or service-user data into a local model.
- If a setup cell errors, read the message and fix the Ollama/model issue before continuing.

The main programming pattern is: pull models, run the same task through each model, score outputs, tune parameters, and make a documented recommendation.


## Setup Preamble

The setup cells below define runtime settings, candidate models, the shared evaluation set, and helper functions.

The notebook pulls missing model weights by default using `AUTO_PULL_MISSING_MODELS = True`. It also attempts to install Ollama from inside the notebook on Linux when `INSTALL_OLLAMA_IF_MISSING = True` and the `ollama` command is missing. On macOS or Windows, use the Ollama installer if the command is not available to the notebook.

Approximate model sizes are shown before downloading. These sizes can change as model tags are updated upstream.


In [ ]:
# Setup 1: imports and runtime settings
import json
import os
import platform
import re
import shutil
import subprocess
import time
import urllib.error
import urllib.request


AUTO_PULL_MISSING_MODELS = True
INSTALL_OLLAMA_IF_MISSING = True
OLLAMA_INSTALL_TIMEOUT_SECONDS = 900
LOCAL_BASE_URL = "http://localhost:11434"
DEFAULT_LOCAL_MODEL = "smollm2:135m"
OLLAMA_COMMAND = "ollama"
OLLAMA_SERVER_PROCESS = None

ALLOWED_LABELS = ["access_barrier", "housing_stress", "health_need", "other"]


In [ ]:
# Setup 2: candidate local models
candidate_models = [
    {
        "model": "smollm2:135m",
        "approx_download_size": "~0.3 GB",
        "size_note": "ultra-small",
        "expected_strength": "lowest memory classroom workflow test",
        "expected_risk": "weakest instruction following",
    },
    {
        "model": "gemma3:270m",
        "approx_download_size": "~0.3 GB",
        "size_note": "ultra-small",
        "expected_strength": "compact instruction-following candidate",
        "expected_risk": "requires a recent Ollama version",
    },
    {
        "model": "qwen2.5:0.5b",
        "approx_download_size": "~0.4 GB",
        "size_note": "small",
        "expected_strength": "often stronger structured-output behaviour than tiny models",
        "expected_risk": "larger than the ultra-small candidates",
    },
]

models_to_prepare = [candidate["model"] for candidate in candidate_models]


In [ ]:
# Setup 3: shared synthetic evaluation set
evaluation_notes = [
    {
        "id": "HS001",
        "text": "Public comment says the evening bus was cancelled and several residents missed diabetes clinic appointments.",
        "human_label": "access_barrier",
    },
    {
        "id": "HS002",
        "text": "Advice note reports rent arrears, threat of eviction, and difficulty reaching the housing office.",
        "human_label": "housing_stress",
    },
    {
        "id": "HS003",
        "text": "Forum post asks for advice after medication side effects and trouble arranging a follow-up appointment.",
        "human_label": "health_need",
    },
    {
        "id": "HS004",
        "text": "Survey response praises new park lighting and weekend community activities.",
        "human_label": "other",
    },
    {
        "id": "HS005",
        "text": "Meeting note says older residents cannot afford taxis to outpatient appointments.",
        "human_label": "access_barrier",
    },
    {
        "id": "HS006",
        "text": "Community worker reports a parent is anxious about rent arrears and a child's missed mental health appointment.",
        "human_label": "housing_stress",
    },
]


In [ ]:
# Setup 4: command and Ollama install helpers
def run_command(command, timeout=1800):
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout,
            check=False,
        )
        return {
            "ok": completed.returncode == 0,
            "returncode": completed.returncode,
            "stdout": completed.stdout.strip(),
            "stderr": completed.stderr.strip(),
        }
    except FileNotFoundError:
        return {"ok": False, "returncode": None, "stdout": "", "stderr": "command not found"}
    except subprocess.TimeoutExpired:
        return {"ok": False, "returncode": None, "stdout": "", "stderr": "command timed out"}


def run_command_live(command, timeout=1800):
    try:
        completed = subprocess.run(command, text=True, timeout=timeout, check=False)
        return {"ok": completed.returncode == 0, "returncode": completed.returncode}
    except FileNotFoundError:
        return {"ok": False, "returncode": None, "stderr": "command not found"}
    except subprocess.TimeoutExpired:
        return {"ok": False, "returncode": None, "stderr": "command timed out"}


def find_ollama_command():
    candidates = [
        shutil.which("ollama"),
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        "/bin/ollama",
        "/Applications/Ollama.app/Contents/Resources/ollama",
    ]

    local_appdata = os.environ.get("LOCALAPPDATA")
    if local_appdata:
        candidates.append(os.path.join(local_appdata, "Programs", "Ollama", "ollama.exe"))

    program_files = os.environ.get("ProgramFiles")
    if program_files:
        candidates.append(os.path.join(program_files, "Ollama", "ollama.exe"))

    candidates.append(os.path.expanduser("~/AppData/Local/Programs/Ollama/ollama.exe"))

    seen = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        if os.path.exists(candidate):
            return candidate

    return None


def ollama_missing_message():
    system_name = platform.system() or "unknown OS"
    return "\n".join(
        [
            "Ollama is not available to this notebook.",
            "",
            f"Detected operating system: {system_name}",
            "",
            "The notebook can attempt an automatic Ollama install on Linux when INSTALL_OLLAMA_IF_MISSING is True.",
            "On macOS or Windows, use the installer from https://ollama.com/download, then restart VS Code/Jupyter.",
            "",
            "After installing, test from this same notebook environment with: !ollama --version",
            "If a terminal can run `ollama --version` but this notebook cannot, launch VS Code from that terminal with `code .` or select the same Python/Jupyter environment.",
            "",
            "Course setup notes: ../setup.md, section 4.",
        ]
    )


def install_ollama_linux():
    if shutil.which("curl") is None:
        raise RuntimeError("Automatic Ollama install needs `curl`, but `curl` is not available.\n" + ollama_missing_message())

    print("Ollama was not found. Attempting Linux install from https://ollama.com/install.sh")
    result = run_command_live(
        ["sh", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        timeout=OLLAMA_INSTALL_TIMEOUT_SECONDS,
    )
    if not result["ok"]:
        raise RuntimeError(
            "Automatic Ollama install failed. It may need internet access or administrator permissions.\n"
            + ollama_missing_message()
        )


def require_ollama_command():
    global OLLAMA_COMMAND

    command = find_ollama_command()
    if command is None and INSTALL_OLLAMA_IF_MISSING and platform.system().lower() == "linux":
        install_ollama_linux()
        command = find_ollama_command()

    if command is None:
        raise RuntimeError(ollama_missing_message())

    OLLAMA_COMMAND = command
    return command


In [ ]:
# Setup 5: Ollama server and model helpers
def ollama_api_is_available(timeout=2):
    try:
        with urllib.request.urlopen(f"{LOCAL_BASE_URL}/api/tags", timeout=timeout) as response:
            return 200 <= response.status < 500
    except Exception:
        return False


def start_ollama_server_if_needed():
    global OLLAMA_SERVER_PROCESS

    if ollama_api_is_available():
        print("Ollama local API is responding.")
        return True

    command = require_ollama_command()
    print("Starting Ollama local server for this notebook...")
    try:
        OLLAMA_SERVER_PROCESS = subprocess.Popen(
            [command, "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    except Exception as exc:
        print(f"Could not start `ollama serve` automatically: {exc}")
        return False

    for _ in range(15):
        time.sleep(1)
        if ollama_api_is_available():
            print("Ollama local API is responding.")
            return True

    print("Could not confirm that the Ollama local API started. The next command will show the detailed error if setup still fails.")
    return False


def parse_ollama_model_names(ollama_list_stdout):
    names = []
    for line in ollama_list_stdout.splitlines():
        stripped = line.strip()
        if not stripped or stripped.lower().startswith("name"):
            continue
        parts = stripped.split()
        if parts:
            names.append(parts[0])
    return names


def list_installed_ollama_models():
    command = require_ollama_command()
    result = run_command([command, "list"], timeout=60)
    if not result["ok"]:
        raise RuntimeError(
            "`ollama list` failed. Start the Ollama app/service, then try again. "
            "If it still fails, run `ollama --version` and `ollama list` in a terminal. "
            f"Details: {result['stderr'] or result['stdout']}"
        )
    return parse_ollama_model_names(result["stdout"])


def model_is_installed(model, installed_models):
    return model in installed_models


def pull_model(model):
    command = require_ollama_command()
    print(f"Pulling {model}. This may take several minutes.")
    result = run_command([command, "pull", model], timeout=3600)
    if not result["ok"]:
        raise RuntimeError(f"`ollama pull {model}` failed. Details: {result['stderr'] or result['stdout']}")
    return result


def ensure_required_models(models, auto_pull=AUTO_PULL_MISSING_MODELS):
    command = require_ollama_command()
    print("Using Ollama command:", command)
    start_ollama_server_if_needed()

    installed_before = list_installed_ollama_models()
    print("Installed models before pull:", installed_before)

    for model in models:
        if model_is_installed(model, installed_before):
            print(f"{model}: already installed")
            continue

        if not auto_pull:
            raise RuntimeError(
                f"{model} is not installed and AUTO_PULL_MISSING_MODELS is False. "
                f"Set AUTO_PULL_MISSING_MODELS = True or run `ollama pull {model}`."
            )

        pull_model(model)

    installed_after = list_installed_ollama_models()
    missing_after = [model for model in models if not model_is_installed(model, installed_after)]
    if missing_after:
        raise RuntimeError(f"These models are still not visible after pulling: {missing_after}")

    print("Installed models after pull:", installed_after)
    return installed_after


In [ ]:
# Setup 6: prompt, API, and output helpers
def make_label_prompt(text, strict=True):
    if strict:
        return f"""
Classify the text as exactly one of:
access_barrier, housing_stress, health_need, other.

Return only the exact snake_case label. Do not use spaces, capitals, punctuation, or explanations.

Text:
{text}
""".strip()

    return f"""
Classify the text as one of:
access_barrier, housing_stress, health_need, other.

Give the label and one short reason.

Text:
{text}
""".strip()


def build_ollama_chat_payload(prompt, model=DEFAULT_LOCAL_MODEL, temperature=0.0, num_predict=30):
    return {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
        },
    }


def post_json(url, payload, timeout=120):
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        start = time.time()
        with urllib.request.urlopen(request, timeout=timeout) as response:
            body = response.read().decode("utf-8")
        elapsed = time.time() - start
        return {"ok": True, "status": response.status, "elapsed_seconds": elapsed, "json": json.loads(body)}
    except urllib.error.HTTPError as exc:
        return {"ok": False, "status": exc.code, "error": exc.read().decode("utf-8", errors="replace")}
    except urllib.error.URLError as exc:
        return {"ok": False, "status": None, "error": str(exc.reason)}
    except TimeoutError:
        return {"ok": False, "status": None, "error": "timeout"}


def extract_ollama_text(result):
    body = result.get("json") or {}
    message = body.get("message") or {}
    return message.get("content") or body.get("response") or ""


def call_ollama_chat(prompt, model=DEFAULT_LOCAL_MODEL, temperature=0.0, num_predict=30, timeout=120):
    payload = build_ollama_chat_payload(prompt, model, temperature, num_predict)
    result = post_json(f"{LOCAL_BASE_URL}/api/chat", payload, timeout=timeout)
    if not result["ok"]:
        raise RuntimeError(
            f"Ollama call failed for model {model}. "
            f"Status: {result.get('status')}. Error: {result.get('error')}"
        )
    return result


def normalise_label(output_text):
    text = (output_text or "").strip().lower()
    text = re.sub(r"^[`'\"\s]+|[`'\"\s]+$", "", text)
    text = re.sub(r"[^a-z_ -].*$", "", text).strip()
    canonical = re.sub(r"[\s-]+", "_", text)
    canonical = re.sub(r"_+", "_", canonical).strip("_")

    label_aliases = {
        "accessbarrier": "access_barrier",
        "access_barriers": "access_barrier",
        "access_barrier": "access_barrier",
        "housingstress": "housing_stress",
        "housing_stresses": "housing_stress",
        "housing_stress": "housing_stress",
        "healthneed": "health_need",
        "health_needs": "health_need",
        "health_need": "health_need",
        "other": "other",
    }

    compact = canonical.replace("_", "")
    if canonical in ALLOWED_LABELS:
        return canonical
    if canonical in label_aliases:
        return label_aliases[canonical]
    if compact in label_aliases:
        return label_aliases[compact]

    for label in ALLOWED_LABELS:
        if canonical.startswith(label) or compact.startswith(label.replace("_", "")):
            return label
    return "invalid"


def run_local_label_task(model, note, temperature=0.0, num_predict=30, strict=True):
    prompt = make_label_prompt(note["text"], strict=strict)
    result = call_ollama_chat(
        prompt=prompt,
        model=model,
        temperature=temperature,
        num_predict=num_predict,
    )
    output_text = extract_ollama_text(result)
    return {
        "model": model,
        "note_id": note["id"],
        "human_label": note["human_label"],
        "temperature": temperature,
        "num_predict": num_predict,
        "strict_prompt": strict,
        "elapsed_seconds": result.get("elapsed_seconds"),
        "output_text": output_text,
        "normalised_label": normalise_label(output_text),
    }



def check_model_can_answer(model):
    start_ollama_server_if_needed()
    try:
        result = call_ollama_chat(
            prompt="Return exactly this label: other",
            model=model,
            temperature=0.0,
            num_predict=8,
            timeout=60,
        )
        text = extract_ollama_text(result).strip()
        if not text:
            return {"ok": False, "error": "model returned an empty response"}
        return {"ok": True, "output_text": text}
    except RuntimeError as exc:
        start_ollama_server_if_needed()
        return {"ok": False, "error": str(exc)}


def select_usable_models(models):
    usable_models = []
    failed_model_checks = {}

    for model in models:
        print(f"Smoke-testing {model}...")
        check = check_model_can_answer(model)
        if check["ok"]:
            print(f"{model}: usable test response -> {check['output_text']}")
            usable_models.append(model)
            continue

        failed_model_checks[model] = check["error"]
        print(f"Skipping {model}: {check['error']}")
        time.sleep(2)

    if not usable_models:
        raise RuntimeError(
            "No local model passed the smoke test. Restart Ollama, rerun the setup cells, "
            "or try pulling a different small Ollama model."
        )

    return usable_models, failed_model_checks


In [ ]:
# Setup 7: scoring helper
def refresh_normalised_labels(rows):
    for row in rows:
        row["normalised_label"] = normalise_label(row.get("output_text", ""))
    return rows


def score_rows(rows):
    rows = refresh_normalised_labels(rows)
    total = len(rows)
    correct = sum(row["normalised_label"] == row["human_label"] for row in rows)
    valid = sum(row["normalised_label"] in ALLOWED_LABELS for row in rows)
    mean_seconds = sum(row.get("elapsed_seconds") or 0 for row in rows) / total if total else None
    return {
        "n": total,
        "accuracy": correct / total if total else 0,
        "valid_label_rate": valid / total if total else 0,
        "mean_elapsed_seconds": mean_seconds,
    }


## Local Model Check and Pull

Run this cell before the comparison sections.

It will:

- find Ollama on PATH or in common app install locations;
- attempt an in-notebook Ollama install on Linux if `INSTALL_OLLAMA_IF_MISSING = True`;
- start the local Ollama server if it is not already responding;
- show the requested models and approximate download sizes;
- run `ollama list`;
- pull any missing model weights because `AUTO_PULL_MISSING_MODELS = True`;
- verify that the required models are visible after pulling;
- smoke-test each pulled model and skip any model whose local runner crashes.

There is no model-free path, but the notebook can continue with a smaller usable model set if one candidate model crashes. If this cell says Ollama is not available on macOS or Windows, install/open Ollama, restart VS Code/Jupyter, and rerun the setup cells plus this cell before continuing. If a terminal can run `ollama --version` but the notebook cannot, start VS Code from that terminal with `code .` or select the same Python/Jupyter environment.


In [ ]:
# Required local runtime and model setup
print("Models required for this lab:")
for candidate in candidate_models:
    print(f"- {candidate['model']} ({candidate['approx_download_size']}): {candidate['size_note']}")

print("\nINSTALL_OLLAMA_IF_MISSING:", INSTALL_OLLAMA_IF_MISSING)
print("AUTO_PULL_MISSING_MODELS:", AUTO_PULL_MISSING_MODELS)
installed_models = ensure_required_models(models_to_prepare)
usable_models, failed_model_checks = select_usable_models(models_to_prepare)
models_to_prepare = usable_models

if failed_model_checks:
    print("\nModels skipped after smoke testing:")
    for model, error in failed_model_checks.items():
        print(f"- {model}: {error}")

print("\nReady to use local models:", models_to_prepare)


## Part A: Set Up a Fair Comparison Task (Approx. 20 minutes)

A model comparison is only useful if each model gets the same task. We will use the same six synthetic notes, the same allowed labels, and the same scoring rule.

The health/social task is intentionally simple: classify each note as `access_barrier`, `housing_stress`, `health_need`, or `other`.


### Question 1: Compare Candidate Local Models

Read the candidate model list. We will compare the prepared models that passed the setup smoke test.

Your task:

- print each candidate model, approximate download size, and expected trade-off;
- set `chosen_models` equal to `models_to_prepare`;
- write one sentence explaining why using the same task across models matters.


In [ ]:
# Answer 1
for candidate in candidate_models:
    print(
        candidate["model"],
        "-",
        candidate["approx_download_size"],
        "-",
        candidate["expected_strength"],
    )

chosen_models = models_to_prepare
reason = "Write your reason here."

print("\nChosen models:", chosen_models)
print("Reason:", reason)


### Question 2: Inspect the Shared Evaluation Set

A fair comparison needs the same examples and the same human labels.

Your task:

- print the note IDs, human labels, and text;
- check how many examples there are;
- check how many examples there are for each label.


In [ ]:
# Answer 2
label_counts = {}

for note in evaluation_notes:
    label = note["human_label"]
    label_counts[label] = label_counts.get(label, 0) + 1
    print(note["id"], label, "-", note["text"])

print("\nNumber of examples:", len(evaluation_notes))
print("Label counts:", label_counts)


## Part B: Compare Model Outputs (Approx. 45 minutes)

Now we run the same notes through each local model. These are real Ollama calls, not simulated outputs.

This section can take time because it makes one local model call for each model-note pair.


### Question 3: Generate Comparable Model Outputs

Use the same strict prompt style and temperature for each model.

Your task:

- loop over the chosen models;
- loop over the evaluation notes;
- call the local model for each model-note pair;
- normalise the output to one of the allowed labels or `invalid`, accepting minor formatting differences such as spaces or title case; later scoring cells refresh these labels if you rerun cells out of order;
- store the rows in `comparison_rows`.


In [ ]:
# Answer 3
comparison_rows = []

for model in chosen_models:
    print(f"\nRunning model: {model}")
    for note in evaluation_notes:
        row = run_local_label_task(
            model=model,
            note=note,
            temperature=0.0,
            num_predict=30,
            strict=True,
        )
        comparison_rows.append(row)
        print(row["note_id"], "->", row["output_text"], "| normalised:", row["normalised_label"])

print("\nRows collected:", len(comparison_rows))


### Question 4: Score Each Candidate Model

A useful comparison needs a small metric table, not only impressions.

Your task:

- group rows by model;
- calculate accuracy;
- calculate valid-label rate;
- inspect mean elapsed time.


In [ ]:
# Answer 4
refresh_normalised_labels(comparison_rows)
model_scores = {}

for model in chosen_models:
    rows_for_model = [row for row in comparison_rows if row["model"] == model]
    model_scores[model] = score_rows(rows_for_model)

print(json.dumps(model_scores, indent=2))


### Question 5: Inspect Disagreements

Accuracy is a summary. Disagreements show what kind of error happened.

Your task:

- print only the rows where the normalised model label differs from the human label;
- comment on whether the errors are concentrated in one model or one kind of note.


In [ ]:
# Answer 5
refresh_normalised_labels(comparison_rows)

disagreements = [
    row for row in comparison_rows
    if row["normalised_label"] != row["human_label"]
]

for row in disagreements:
    print(row)

print("\nComment: write one sentence about the disagreement pattern.")


## Part C: Tune Parameters and Prompt Style (Approx. 35 minutes)

Local models are not only compared by model name. Their behaviour also changes with generation parameters and prompt wording.

For beginner work, focus on three controls:

- `temperature`: lower values are usually better for consistent labels;
- `num_predict`: shorter outputs reduce rambling for label-only tasks;
- prompt style: strict label-only prompts are easier to score than explanation prompts.


### Question 6: Build a Small Parameter Grid

A parameter grid is a list of settings to test. Keep it small because every setting makes multiple local model calls.

Your task:

- choose one model to tune;
- test two temperatures;
- keep output length short.


In [ ]:
# Answer 6
tuning_model = chosen_models[0]

parameter_grid = [
    {"temperature": 0.0, "num_predict": 20, "strict": True},
    {"temperature": 0.5, "num_predict": 20, "strict": True},
]

for setting in parameter_grid:
    print(setting)


### Question 7: Run the Parameter Grid

Use the same model and same evaluation notes under each parameter setting.

Your task:

- loop over `parameter_grid`;
- call the local model;
- score each setting;
- identify which setting is most reliable for label-only classification.


In [ ]:
# Answer 7
tuning_results = []

for setting in parameter_grid:
    print(f"\nTesting setting: {setting}")
    rows = []
    for note in evaluation_notes:
        row = run_local_label_task(
            model=tuning_model,
            note=note,
            temperature=setting["temperature"],
            num_predict=setting["num_predict"],
            strict=setting["strict"],
        )
        rows.append(row)
        print(row["note_id"], "->", row["output_text"], "| normalised:", row["normalised_label"])

    score = score_rows(rows)
    tuning_results.append({"setting": setting, "score": score})

print("\nTuning results:")
print(json.dumps(tuning_results, indent=2))


### Question 8: Compare Strict and Explanation Prompts

Sometimes we want the model to explain itself. But explanations make automatic scoring harder and can encourage unsupported reasoning.

Your task:

- use one note;
- run one strict prompt and one explanation prompt;
- normalise both outputs;
- explain why the strict output is easier to evaluate.


In [ ]:
# Answer 8
example_note = evaluation_notes[0]
prompt_test_model = chosen_models[0]

strict_row = run_local_label_task(
    model=prompt_test_model,
    note=example_note,
    temperature=0.0,
    num_predict=30,
    strict=True,
)

explanation_row = run_local_label_task(
    model=prompt_test_model,
    note=example_note,
    temperature=0.0,
    num_predict=80,
    strict=False,
)

print("Strict output:", strict_row["output_text"])
print("Strict normalised:", strict_row["normalised_label"])
print("Explanation output:", explanation_row["output_text"])
print("Explanation normalised:", explanation_row["normalised_label"])
print("Comment: write one sentence here.")


## Part D: Final Recommendation (Approx. 20 minutes)

Finish by combining quality, valid-label rate, and speed into a recommendation.


### Question 9: Recommend a Model and Setting

Use your real local outputs to choose a configuration.

Your task:

- choose the best model from `model_scores`;
- choose the best setting from `tuning_results`;
- explain the trade-off.


In [ ]:
# Answer 9
best_model = max(model_scores, key=lambda model: (model_scores[model]["accuracy"], model_scores[model]["valid_label_rate"]))
best_setting = max(tuning_results, key=lambda item: (item["score"]["accuracy"], item["score"]["valid_label_rate"]))

print("Best model:", best_model)
print("Best model score:", model_scores[best_model])
print("Best setting:", best_setting)
print("Trade-off: write one sentence here.")


### Question 10: Final Comparison Report

Write a compact model-comparison report.

Your report should include:

- the task;
- the candidate models;
- the best model/settings combination;
- the evidence used;
- one limitation;
- one governance note for health/social use.


In [ ]:
# Answer 10
final_report = {
    "task": "classify synthetic health/social notes",
    "candidate_models": chosen_models,
    "recommended_model": best_model,
    "recommended_setting": best_setting["setting"],
    "evidence": "Use your model score table and parameter grid results here.",
    "limitation": "Write one limitation here.",
    "governance_note": "Use synthetic or approved de-identified data only.",
}

print(json.dumps(final_report, indent=2))


## End of Lab Checklist

By the end of this lab, you should have:

- checked that Ollama is available to the notebook;
- pulled all required local models by default;
- verified that the required models are visible to `ollama list`;
- smoke-tested local models before comparison and skipped any crashing candidates;
- used one shared health/social evaluation set;
- called real local models for all comparison outputs;
- normalised model outputs to allowed labels;
- scored accuracy, valid-label rate, and elapsed time;
- inspected disagreements rather than relying only on accuracy;
- tested a small parameter grid;
- compared strict prompts with explanation prompts;
- written a final model-comparison recommendation.
